[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-survival.ipynb)

# Survival Analysis

*AIBits Academy · Machine Learning End To End · Specialized Supervised Learning · New*

A family of techniques purpose-built for "time until an event happens" — and for the very common, tricky problem of subjects who haven't experienced the event yet when you collect your data.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q lifelines

> **🎯 Intuition First**
>
> The whole field turns on one refusal: **don't throw away the people who haven't had the event yet.** An employee still on payroll hasn't told you nothing — they've told you their tenure is *at least* this long. Survival analysis is the machinery that squeezes that partial fact for all it's worth, instead of discarding it.

## Why Not Just Use Regression on "Time to Event"?

Imagine predicting "months until an employee resigns" at a Bengaluru tech company using ordinary regression. The problem: many employees in your dataset are *still employed* at the moment you collect data — you don't know their eventual tenure, only that it's *at least* as long as their tenure so far. Simply dropping these still-employed rows discards your most loyal employees and severely biases the model toward short tenures. This partial information is called **censoring**, and survival analysis is the statistical machinery purpose-built to use it correctly rather than discard it.

**Right-censoring** (the common case): we know the event hasn't happened by the end of observation, but not when it eventually will (or if it ever will). An employee still on payroll after 3 years is right-censored at t=3 — their true resignation time is ≥ 3 years, unknown exactly.

## The Survival Function

$$S(t) = P(T > t) \qquad \text{--- the probability an individual "survives" (hasn\'t experienced the event) past time } t$$

S(t) starts at 1 (everyone alive/employed/active at t=0) and decreases monotonically toward 0 (or toward whatever fraction never experiences the event) as t grows.

## Kaplan-Meier Estimator — Non-Parametric Survival Curves

The Kaplan-Meier estimator computes S(t) directly from the data, without assuming any particular distributional shape, by multiplying the conditional survival probability at each observed event time:

$$\hat{S}(t) = \prod_{i:\,t_i\le t}\left(1-\frac{d_i}{n_i}\right) \quad \text{where } n_i = \text{at-risk count just before } t_i,\ d_i = \text{events at } t_i$$

## Kaplan-Meier Curve — Employee Attrition (animated)

Press play to watch Ŝ(t) built *event by event* from the 12-employee cohort in the code below. The curve steps down only at real resignations (purple dots); censored, still-employed staff (green ticks) never pull it down — they just leave the at-risk pool. The dashed amber guide marks the median tenure, where Ŝ(t) first reaches 0.5.

The curve only steps *down* at actual resignation events — censored employees (still employed, marked with tick marks) don't pull the curve down; they simply stop contributing to the "at-risk" count from that point onward. This is precisely how Kaplan-Meier correctly uses partial information instead of discarding it.

## Code — Kaplan-Meier From Scratch (Pyodide-Compatible)

The `lifelines` library is the standard tool for this in real projects, but it isn't available in-browser via Pyodide — so here's the estimator implemented directly with NumPy, exactly the logic the Practice Hub sandbox will run:

In [ ]:
import numpy as np

# Employee tenure data (months) — event=1 resigned, event=0 still employed (censored)
tenure = np.array([4, 7, 9, 9, 13, 18, 18, 22, 26, 31, 33, 36])
event  = np.array([1, 1, 0, 1, 1,  0,  1,  1,  0,  1,  1,  0])

def kaplan_meier(tenure, event):
    order = np.argsort(tenure)
    tenure, event = tenure[order], event[order]
    n_at_risk = len(tenure)
    S = 1.0
    curve = []
    for t in np.unique(tenure):
        mask = tenure == t
        d = event[mask].sum()          # events at this exact time
        n = n_at_risk                   # at-risk just before time t
        if d > 0:
            S *= (1 - d/n)
        curve.append((t, S))
        n_at_risk -= mask.sum()         # everyone at time t leaves the risk set
    return curve

for t, s in kaplan_meier(tenure, event):
    print(f"  t={t:2d} months   Ŝ(t)={s:.3f}")

# Median survival time: first t where Ŝ(t) drops to ≤ 0.5
curve = kaplan_meier(tenure, event)
median_t = next(t for t, s in curve if s <= 0.5)
print(f"\nMedian tenure (50% still employed threshold): {median_t} months")

## Cox Proportional Hazards — Adding Covariates

Kaplan-Meier answers "what does survival look like overall (or per group)?" but can't tell you *which features drive attrition risk*. The Cox Proportional Hazards model adds covariates to a hazard function h(t) — the instantaneous risk of the event at time t, given survival up to t:

$$h(t\mid x) = h_0(t)\cdot \exp(\mathbf{x}^{\top}\beta)$$

h₀(t) is an unspecified **baseline hazard** (Cox's key innovation — it needn't be estimated to get valid coefficient estimates, hence "semi-parametric"). exp(βⱼ) is the **hazard ratio** for feature xⱼ: a hazard ratio of 1.4 for "long commute" means employees with long commutes resign at 1.4× the instantaneous rate of otherwise-identical employees, at every point in time (the "proportional hazards" assumption — the ratio is constant, not time-varying).

In [ ]:
# Conceptual usage with the standard `lifelines` library (not Pyodide-available, shown for reference)
from lifelines import CoxPHFitter
import pandas as pd

df = pd.DataFrame({
    'tenure_months': tenure, 'resigned': event,
    'long_commute': [1,0,1,0,1,0,1,0,0,1,1,0],
    'remote_eligible': [0,0,1,1,0,1,0,0,1,0,0,1],
})
cph = CoxPHFitter()
cph.fit(df, duration_col='tenure_months', event_col='resigned')
cph.print_summary()
# hazard ratio (exp(coef)) > 1 → increases resignation risk
# hazard ratio < 1 → protective (reduces resignation risk)

## When to Reach for Survival Analysis

| Business question | Event | Censoring source |
|---|---|---|
| Employee attrition (HR analytics) | Resignation | Still employed at data pull |
| Loan default timing (NBFC risk) | Default | Loan still active / fully repaid without default |
| Customer churn timing (subscription) | Cancellation | Still subscribed at data pull |
| Equipment failure (manufacturing) | Breakdown | Machine still running at inspection |
| Clinical trials (pharma) | Disease recurrence / death | Patient lost to follow-up or trial ended first |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How much of the data is censored?

`event = 1` means the employee left; `0` means still employed (censored). Store the **fraction censored** in `frac_censored`.

In [ ]:
import numpy as np
event = np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0])
frac_censored = None   # TODO


In [ ]:
try:
    check("one third is censored", abs(frac_censored - 1 / 3) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
event = np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0])
frac_censored = float((event == 0).mean())

```

</details>

### Exercise 2 · Medium · Median survival with lifelines

Fit a `KaplanMeierFitter` on the 12-employee data and store the median survival time in `median_months` (the time at which the curve first drops to 0.5 or below).

In [ ]:
import numpy as np
from lifelines import KaplanMeierFitter
tenure = np.array([4, 7, 9, 9, 13, 18, 18, 22, 26, 31, 33, 36])
event = np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0])
median_months = None   # TODO


In [ ]:
try:
    check("median is a sensible tenure", 13 <= median_months <= 36)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from lifelines import KaplanMeierFitter
tenure = np.array([4, 7, 9, 9, 13, 18, 18, 22, 26, 31, 33, 36])
event = np.array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0])
kmf = KaplanMeierFitter().fit(tenure, event_observed=event)
median_months = kmf.median_survival_time_

```

</details>

### Exercise 3 · Stretch · Kaplan–Meier by hand

Write `km(tenure, event)` returning a list of `(time, survival)` for each time at which an event occurs. At each event time: `S *= 1 - d/n` where `d` = events at that time and `n` = people still at risk. Censored people leave the risk set **after** that time.

In [ ]:
def km(tenure, event):
    pass   # TODO


In [ ]:
try:
    out = km([2, 3, 3, 5, 8], [1, 1, 0, 1, 1])
    check("hand-worked example", [(t, round(s, 4)) for t, s in out] == [(2, 0.8), (3, 0.6), (5, 0.3), (8, 0.0)])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def km(tenure, event):
    pairs = sorted(zip(tenure, event))
    times = sorted({t for t, e in pairs if e == 1})
    s, out = 1.0, []
    for t in times:
        n = sum(1 for tt, _ in pairs if tt >= t)
        d = sum(1 for tt, e in pairs if tt == t and e == 1)
        s *= 1 - d / n
        out.append((t, s))
    return out

```

Censored people still count in the denominator up to their last observed time — that is how KM uses partial information.

</details>

---
*Back to the course: **Machine Learning End To End → Survival Analysis**.*